# PyTorch Classifier — Reusable Template

**Short name:** `PTIntro` template. Copy this notebook, point `DATA_PATH` at your table, edit `FEATURE_COLS` / `TARGET_COL` / `CLASS_ORDER`, then run.

Contract that stays fixed:

1. Features `float32`, class indices `long`.
2. `nn.Module` with a ReLU hidden layer, logits out.
3. `CrossEntropyLoss` + Adam (swap the optimizer in one line).
4. `model.eval()` + `torch.no_grad()` before you score.
5. A simulation cell with `hidden` / `lr` / `epochs` / `label_noise`.


## Inline cheat-sheet (keep this cell visible)

See also **`PTIntro_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Tensor ranks | scalar 0-D, vector 1-D, matrix 2-D, batch of images 4-D `(N,C,H,W)` |
| From a list | `torch.tensor([[1,2],[3,4]])` — always copies |
| From NumPy | `torch.from_numpy(a)` shares memory; `torch.as_tensor(a)` may share |
| Zeros / ones | `torch.zeros(2,3)`, `torch.ones_like(x)` |
| Sequences | `torch.arange(start, end, step)` end **exclusive**; `torch.linspace(a,b,steps)` both ends **inclusive** |
| dtypes | features `float32`; class indices `long` / `torch.int64` |
| Device | `x.to("cuda")` if a GPU is present; this lab stays on CPU |
| Module | subclass `nn.Module`, call `super().__init__()`, store layers as `self.*` |
| Forward | `model(x)` — **not** `model.forward(x)` — so hooks run |
| ReLU hidden | `F.relu(self.fc1(x))` then linear logits |
| Loss | `nn.CrossEntropyLoss()` = log-softmax + NLL. Do **not** softmax first |
| Train step | `zero_grad()` → forward → loss → `backward()` → `step()` |
| Eval | `model.eval()` **and** `with torch.no_grad():` |
| Labels | `torch.argmax(logits, dim=1)` |

**Order that matters:** `zero_grad` before `backward`. `eval()` does not turn off autograd by itself.


## Drop-in training script


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

DATA_PATH = "data/iris.csv"
FEATURE_COLS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
TARGET_COL = "species"
CLASS_ORDER = ["setosa", "versicolor", "virginica"]  # labels become 0..K-1
HIDDEN = 16
LR = 0.01
EPOCHS = 100
TEST_SIZE = 0.2
SEED = 42

df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLS].to_numpy(np.float32)
y = df[TARGET_COL].map({c: i for i, c in enumerate(CLASS_ORDER)}).to_numpy(np.int64)
assert not np.isnan(y).any(), "CLASS_ORDER does not cover every label"

def split(X, y, test_size=0.2, seed=42):
    rng = np.random.RandomState(seed)
    tr, te = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n_te = int(round(len(idx) * test_size))
        te.append(idx[:n_te]); tr.append(idx[n_te:])
    tr, te = np.concatenate(tr), np.concatenate(te)
    rng.shuffle(tr); rng.shuffle(te)
    return X[tr], X[te], y[tr], y[te]

Xtr, Xte, ytr, yte = split(X, y, TEST_SIZE, SEED)
Xtr_t, Xte_t = torch.tensor(Xtr), torch.tensor(Xte)
ytr_t, yte_t = torch.tensor(ytr, dtype=torch.long), torch.tensor(yte, dtype=torch.long)

class MLP(nn.Module):
    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_h)
        self.fc2 = nn.Linear(d_h, d_out)
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

torch.manual_seed(SEED)
model = MLP(X.shape[1], HIDDEN, len(CLASS_ORDER))
opt = torch.optim.Adam(model.parameters(), lr=LR)
crit = nn.CrossEntropyLoss()
losses = []
for ep in range(EPOCHS):
    opt.zero_grad()
    loss = crit(model(Xtr_t), ytr_t)
    loss.backward(); opt.step()
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    pred = model(Xte_t).argmax(1)
acc = (pred == yte_t).float().mean().item()
print(f"test acc {acc*100:.2f}%  final loss {losses[-1]:.4f}")
plt.plot(losses); plt.title("loss"); plt.xlabel("epoch"); plt.show()


## How to reuse

1. Point `DATA_PATH` at a tidy CSV.
2. List numeric feature columns. Encode the target as contiguous integers `0..K-1`.
3. Keep `CrossEntropyLoss` for mutually exclusive classes. For a 0/1 probability head use `BCEWithLogitsLoss` and a single output unit instead.
4. When *n* > a few thousand, wrap tensors in `DataLoader` and step per mini-batch.
5. Scale features (`StandardScaler` or a train-only mean/sd) before the first `Linear` once columns live on different units.
6. Simulation knobs that transfer: hidden width, learning rate, epochs, label noise, train-set size.

Short name stays `PTIntro` so GitHub paths do not blow the limit.
